# What Causes U.S. Flight Delays?
## Randall Sutton

## Overview

**Question:** What causes flight delays? Do they differ by airline or region?

**Data:** 2 million U.S. flights (2005-2020)

**Key Findings:**
1. Late aircraft is the biggest cause of delays
2. Airlines have different delay profiles
3. Delay patterns are consistent across regions
4. Delays got worse over time (until 2020)

In [ ]:
# Setup - imports and data loading
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb

%matplotlib inline

import warnings
warnings.simplefilter("ignore")

# Helper functions
def clean_cause_names(df):
    """Remove 'Delay' suffix from column names."""
    df = df.copy()
    df.columns = [c.replace('Delay', '') for c in df.columns]
    return df

def get_top_n(df, column, n=10):
    """Get top N values by frequency."""
    return df[column].value_counts().head(n).index

In [ ]:
# Load and prepare data
airline_data = pd.read_csv("data-visualization/airline_2m.csv", encoding="latin-1", low_memory=False)

cols = ['Year', 'Month', 'Reporting_Airline', 'OriginStateName',
        'ArrDelay', 'ArrDelayMinutes', 'ArrDel15',
        'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']

df = airline_data[cols].copy()
df = df[(df['Year'] >= 2005) & (df['Year'] <= 2020)]

delay_causes = ['LateAircraftDelay', 'CarrierDelay', 'NASDelay', 'WeatherDelay', 'SecurityDelay']
delayed = df.dropna(subset=delay_causes).copy()

# Set consistent color palette
cause_colors = sb.color_palette('Set2', 5)

## Main Delay Causes

Late Aircraft delays contribute the most minutes on average, followed by Carrier and NAS delays. Weather and Security are minor factors.

In [ ]:
# Visualization 1: Horizontal bar chart of average delay minutes by cause
avg_delays = delayed[delay_causes].mean().sort_values(ascending=True)
avg_delays.index = [c.replace('Delay', '') for c in avg_delays.index]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(avg_delays.index, avg_delays.values, color=sb.color_palette('Set2', 5))

# Add value labels
for bar, val in zip(bars, avg_delays.values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.1f}', 
            va='center', fontsize=12, fontweight='bold')

ax.set_xlabel('Average Delay (minutes)', fontsize=12)
ax.set_ylabel('Delay Cause', fontsize=12)
ax.set_title('Average Minutes by Delay Cause', fontsize=14, fontweight='bold')
ax.set_xlim(0, avg_delays.max() * 1.15)
ax.tick_params(axis='both', labelsize=11)
plt.tight_layout()
plt.show()

## Delays by Airline

Airlines show distinct delay profiles. Some are dominated by Late Aircraft delays, while others have more balanced contributions from Carrier and NAS delays.

In [ ]:
# Visualization 2: Faceted bar chart showing delay causes for top 6 airlines
top_6 = get_top_n(df, 'Reporting_Airline', 6)
facet_data = delayed[delayed['Reporting_Airline'].isin(top_6)].copy()

# Melt delay causes into long format
facet_melted = facet_data[['Reporting_Airline'] + delay_causes].melt(
    id_vars='Reporting_Airline', var_name='Cause', value_name='Minutes')
facet_melted['Cause'] = facet_melted['Cause'].str.replace('Delay', '')

g = sb.FacetGrid(facet_melted, col='Reporting_Airline', col_wrap=3,
                 height=3.5, sharey=True, aspect=1.1)
g.map_dataframe(sb.barplot, x='Cause', y='Minutes', hue='Cause',
                palette='Set2', errorbar=None, legend=False,
                order=['LateAircraft', 'Carrier', 'NAS', 'Weather', 'Security'])
g.set_titles('{col_name}', fontsize=12, fontweight='bold')
g.set_xticklabels(rotation=45, ha='right', fontsize=9)
g.set_axis_labels('Delay Cause', 'Avg Delay (min)')
g.fig.subplots_adjust(top=0.88)
g.fig.suptitle('Average Delay by Cause — Top 6 Airlines', fontsize=14, fontweight='bold')
plt.show()

## Delays by Region

Delay cause profiles are similar across the busiest states. Late Aircraft and Carrier delays dominate in all regions, with Weather and Security remaining minor contributors.

In [ ]:
# Visualization 3: Clustered bar chart of delays by state
top_10_states = get_top_n(df, 'OriginStateName', 10)
state_causes = (delayed[delayed['OriginStateName'].isin(top_10_states)]
                .groupby('OriginStateName')[delay_causes]
                .mean())
state_causes = clean_cause_names(state_causes)

# Sort by total delay
state_causes['Total'] = state_causes.sum(axis=1)
state_causes = state_causes.sort_values('Total', ascending=False).drop('Total', axis=1)

fig, ax = plt.subplots(figsize=(12, 6))
state_causes.plot(kind='bar', ax=ax, width=0.8, colormap='Set2', edgecolor='white')
ax.set_ylabel('Average Delay (minutes)', fontsize=12)
ax.set_xlabel('Origin State', fontsize=12)
ax.set_title('Average Delay Minutes by Cause and State (Top 10 States)', fontsize=14, fontweight='bold')
ax.legend(title='Cause', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
ax.tick_params(axis='x', rotation=45, labelsize=10)
ax.tick_params(axis='y', labelsize=10)
plt.tight_layout()
plt.show()

## Delays Over Time

Average delay minutes increased from 2005 to 2019. Late Aircraft delays grew the most, driving the overall increase. 2020 shows a notable dip.

In [ ]:
# Visualization 4: Stacked area chart of delay causes over time
yearly_causes = delayed.groupby('Year')[delay_causes].mean()
yearly_causes = clean_cause_names(yearly_causes)

fig, ax = plt.subplots(figsize=(12, 6))
yearly_causes.plot(kind='area', stacked=True, ax=ax, alpha=0.8, colormap='Set2', linewidth=1)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Average Delay (minutes)', fontsize=12)
ax.set_title('Delay Cause Trends Over Time (2005-2020)', fontsize=14, fontweight='bold')
ax.legend(title='Cause', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
ax.set_xlim(2005, 2020)
ax.tick_params(axis='both', labelsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Conclusions

1. **Late Aircraft delays are the main problem** — contributing the most delay minutes on average

2. **Airlines differ** — each carrier has a distinct delay cause profile

3. **Regions are similar** — delay cause patterns are consistent across states

4. **Getting worse over time** — average delays increased from 2005 to 2019, driven by Late Aircraft growth

In [ ]:
# Generate slideshow command (run from terminal):
# jupyter nbconvert Part_II_slide_deck_template.ipynb --to slides --post serve --no-input --no-prompt